# Sports Betting Arbitrage Scraper (Novibet, Stoiximan, Efbet)
This notebook scrapes odds from three major providers using direct deep links, cleans the data, and identifies arbitrage opportunities.

In [ ]:
import sys
sys.path.append('..')

# Import existing modules
try:
    from web_scrape_functions import novibet_functions as nv
    from web_scrape_functions import stoiximan_function as stm
    from web_scrape_functions import efbet_function as ef  # <--- NEW: Efbet Import
except ImportError:
    print("Warning: web_scrape_functions folder or modules not found.")

import pandas as pd
import duckdb
import time
import re
from unidecode import unidecode
from fuzzywuzzy import fuzz
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

In [ ]:
# --- SETUP DRIVER ---
options = webdriver.ChromeOptions()
# options.add_argument("--headless") # Uncomment for background run
options.add_argument("--window-size=1920,1200")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--no-sandbox")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 15)

## 1. Scraping Data

In [ ]:
# --- 1.1 NOVIBET ---
print("--- Starting Novibet ---")
try:
    page_url = 'https://www.novibet.gr/en/sports'
    
    # Football
    football_string = nv.novibet_football_text(page_url, driver)
    nv.novibet_football_export(football_string)
    print("Novibet Football saved.")

    # Basketball
    basketball_string = nv.novibet_basketball_text(driver)
    nv.novibet_basketball_export(basketball_string)
    print("Novibet Basketball saved.")
    
except Exception as e:
    print(f"Novibet Error: {e}")

In [ ]:
# --- 1.2 STOIXIMAN ---
print("\n--- Starting Stoiximan ---")
try:
    # Football
    football_url = 'https://en.stoiximan.gr/sport/soccer/'
    football_string_stm = stm.stoiximan_football_text(football_url, driver)
    stm.stoiximan_football_export(football_string_stm)
    print("Stoiximan Football saved.")

    # Basketball
    basketball_url = 'https://en.stoiximan.gr/sport/basketball/'
    basketball_string_stm = stm.stoiximan_basketball_text(basketball_url, driver)
    stm.stoiximan_basketball_export(basketball_string_stm)
    print("Stoiximan Basketball saved.")
    
except Exception as e:
    print(f"Stoiximan Error: {e}")

In [ ]:
# --- 1.3 EFBET ---
print("\n--- Starting Efbet ---")
try:
    # Football
    football_string_ef = ef.efbet_football_text(driver)
    ef.efbet_football_export(football_string_ef)
    print("Efbet Football saved.")

    # Basketball
    basketball_string_ef = ef.efbet_basketball_text(driver)
    ef.efbet_basketball_export(basketball_string_ef)
    print("Efbet Basketball saved.")
    
except Exception as e:
    print(f"Efbet Error: {e}")

# Close driver after scraping is done
driver.quit()

## 2. Data Loading & Cleaning

In [ ]:
def remove_unicode(df):
    return df.apply(lambda x: unidecode(x) if isinstance(x, str) else x)

def clean_df(df):
    if df is None or df.empty: return pd.DataFrame()
    # Standardize columns
    col_map = {'One_odd': '1', 'X_odd': 'X', 'Two_odd': '2', 'O_odd': 'O_odds', 'U_odd': 'U_odds'}
    df.rename(columns=col_map, inplace=True)
    
    # Clean text columns
    for col in ['Team1', 'Team2']:
        if col in df.columns:
            df[col] = remove_unicode(df[col].astype(str)).str.lower()
            df[col] = df[col].apply(lambda x: ' '.join([w for w in x.split() if len(w)>2]))
    return df

# Load Data
try:
    df_novi = clean_df(pd.read_csv('data/novibet_football.csv'))
    df_stoi = clean_df(pd.read_csv('data/stoiximan_football.csv'))
    df_efbet = clean_df(pd.read_csv('data/efbet_football.csv'))
except FileNotFoundError:
    print("One or more CSV files are missing. Run scraping cells first.")
    df_novi, df_stoi, df_efbet = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

## 3. SQL Queries (3-Way Arbitrage)

In [ ]:
dbcon = duckdb.connect()

# Register Tables
if not df_novi.empty: dbcon.register('table1', df_novi)
if not df_stoi.empty: dbcon.register('table2', df_stoi)
if not df_efbet.empty: dbcon.register('table3', df_efbet)

# --- 3-Way Over/Under Query ---
query_over_under = """
WITH joined_data AS (
    SELECT 
        t1.Team1, t1.Team2,
        t1.O_odds as O_novi, t1.U_odds as U_novi,
        t2.O_odds as O_stoi, t2.U_odds as U_stoi,
        t3.O_odds as O_ef, t3.U_odds as U_ef
    FROM table1 t1 
    INNER JOIN table2 t2 ON t1.Team1 = t2.Team1 
    INNER JOIN table3 t3 ON t1.Team1 = t3.Team1 
    WHERE t1.O_odds IS NOT NULL AND t2.O_odds IS NOT NULL AND t3.O_odds IS NOT NULL
),
calc_max AS (
    SELECT 
        *,
        GREATEST(O_novi, O_stoi, O_ef) as O_max,
        GREATEST(U_novi, U_stoi, U_ef) as U_max
    FROM joined_data
),
calc_arb AS (
    SELECT *, (1/O_max + 1/U_max) as arb
    FROM calc_max
)
SELECT * FROM calc_arb WHERE arb < 1.00 ORDER BY arb ASC;
"""

# --- 3-Way 1X2 Query ---
query_1x2 = """
WITH joined_data AS (
    SELECT 
        t1.Team1, t1.Team2,
        t1."1" as '1_novi', t1.X as 'X_novi', t1."2" as '2_novi',
        t2."1" as '1_stoi', t2.X as 'X_stoi', t2."2" as '2_stoi',
        t3."1" as '1_ef', t3.X as 'X_ef', t3."2" as '2_ef'
    FROM table1 t1
    INNER JOIN table2 t2 ON t1.Team1 = t2.Team1
    INNER JOIN table3 t3 ON t1.Team1 = t3.Team1
    WHERE t1."1" IS NOT NULL AND t2."1" IS NOT NULL AND t3."1" IS NOT NULL
),
calc_max AS (
    SELECT 
        *,
        GREATEST("1_novi", "1_stoi", "1_ef") as "1_max",
        GREATEST("X_novi", "X_stoi", "X_ef") as "X_max",
        GREATEST("2_novi", "2_stoi", "2_ef") as "2_max"
    FROM joined_data
),
calc_arb AS (
    SELECT *, (1/"1_max" + 1/"X_max" + 1/"2_max") as arb
    FROM calc_max
)
SELECT * FROM calc_arb WHERE arb < 1.00 ORDER BY arb ASC;
"""

# Execute
try:
    print("--- Over/Under Opportunities (Exact Match) ---")
    df_ou = dbcon.query(query_over_under).to_df()
    display(df_ou)
    
    print("\n--- 1X2 Opportunities (Exact Match) ---")
    df_1x2 = dbcon.query(query_1x2).to_df()
    display(df_1x2)
except Exception as e:
    print(f"SQL Error (tables might be empty): {e}")

## 4. Fuzzy Matching (3-Way Advanced)
Uses string similarity to find matches that SQL exact join might miss.

In [ ]:
matches = []
print("Running Fuzzy Match...")

# Iterate through Novibet (Reference)
for index, row in df_novi.iterrows():
    t1_novi = row['Team1']
    t2_novi = row['Team2']
    
    # 1. Find Stoiximan Match
    stoi_match = df_stoi[
        (df_stoi['Team1'].apply(lambda x: fuzz.token_sort_ratio(x, t1_novi)) > 80) &
        (df_stoi['Team2'].apply(lambda x: fuzz.token_sort_ratio(x, t2_novi)) > 80)
    ]
    
    # 2. Find Efbet Match
    ef_match = df_efbet[
        (df_efbet['Team1'].apply(lambda x: fuzz.token_sort_ratio(x, t1_novi)) > 80) &
        (df_efbet['Team2'].apply(lambda x: fuzz.token_sort_ratio(x, t2_novi)) > 80)
    ]
    
    # Only proceed if we found a match in AT LEAST one other bookmaker
    if not stoi_match.empty or not ef_match.empty:
        # Get Odds (default to 0 if missing)
        o_novi = float(row.get('O_odds', 0))
        u_novi = float(row.get('U_odds', 0))
        
        o_stoi = float(stoi_match.iloc[0]['O_odds']) if not stoi_match.empty else 0
        u_stoi = float(stoi_match.iloc[0]['U_odds']) if not stoi_match.empty else 0
        
        o_ef = float(ef_match.iloc[0]['O_odds']) if not ef_match.empty else 0
        u_ef = float(ef_match.iloc[0]['U_odds']) if not ef_match.empty else 0
        
        # Calculate Max Odds
        max_o = max(o_novi, o_stoi, o_ef)
        max_u = max(u_novi, u_stoi, u_ef)
        
        if max_o > 1 and max_u > 1:
            arb = (1/max_o) + (1/max_u)
            
            if arb < 1.0: # Arbitrage found!
                matches.append({
                    'Match': f"{t1_novi} vs {t2_novi}",
                    'Arb %': round(arb * 100, 2),
                    'Max Over': max_o,
                    'Max Under': max_u,
                    'O Bookie': 'Novibet' if max_o == o_novi else ('Stoiximan' if max_o == o_stoi else 'Efbet'),
                    'U Bookie': 'Novibet' if max_u == u_novi else ('Stoiximan' if max_u == u_stoi else 'Efbet')
                })

# Create DataFrame
df_fuzzy_arbs = pd.DataFrame(matches).sort_values(by='Arb %')
print(f"Found {len(df_fuzzy_arbs)} fuzzy matches with arbitrage.")
display(df_fuzzy_arbs)